In [1]:
!pip install wurlitzer
from wurlitzer import sys_pipes_forever

sys_pipes_forever()

In [2]:
#using uproot to write generated event into TTree

!pip install uproot
import uproot

import numpy as np
import pandas as pd



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.5/397.5 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 919.6/919.6 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 656.7/656.7 kB 44.9 MB/s eta 0:00:00


In [3]:
#create a class for jets, to overload the __add__ operation

class MJet:
  """
  class for jets, containing 4-momenta and other useful information for jet clustering;
  argument is simply 4-momenta (E,px,py,pz)
  makes use of NumPy for vectorized operations, and other math operations
  """

  def __init__(self, E, px, py, pz):
    self.E = E
    self.px = px
    self.py = py
    self.pz = pz



    self.pT = float(np.sqrt(px**2 + py**2))
    self.p = float(np.sqrt(px**2 + py**2 + pz**2))

    self.phi = float(np.arctan2(self.py, self.px))

    # Calculate mass, add tolerance (1e-6)for floating point issues
    m_squared = self.E**2 - self.p**2
    if m_squared < 0 and np.isclose(m_squared, 0, atol=1e-6): # Treat small negative as zero
        self.m = 0.0
    else:
        self.m = float(np.sqrt(m_squared))

                                                                                                                                                    # Calculate eta, handling edge cases for rapidity
                                                                                                                                                    # maybe also use np.isclose()
    if self.E + self.pz == 0:
        self.eta = -np.inf
    elif self.E - self.pz == 0:
        self.eta = np.inf
    else:
        self.eta = (1/2)*float( np.log( (self.E + self.pz) / (self.E - self.pz) ) )


  def __str__(self):

    output = {
        "E": self.E,
        "px": self.px,
        "py": self.py,
        "pz": self.pz,
        "pT": self.pT,
        "p": self.p,
        "phi": self.phi,
        "eta": self.eta,
        "m": self.m
    }
    return str(output)


  def __add__(self, other):
    return MJet(self.E + other.E, self.px + other.px, self.py + other.py, self.pz + other.pz)

  def __eq__(self, other):
    return (self.E == other.E) and (self.px == other.px) and (self.py == other.py) and (self.pz == other.pz)





In [4]:
#read event data

file4 = uproot.open("PythiaEventsBatchTest.root")

tree = file4['pdEventTree']

#read branches into Awkward array, then numpy arrays

E_array = tree['E'].array().to_numpy()
px_array = tree['px'].array().to_numpy()
py_array = tree['py'].array().to_numpy()
pz_array = tree['pz'].array().to_numpy()
eta_array = tree['eta'].array().to_numpy()
phi_array = tree['phi'].array().to_numpy()



In [5]:
# making the jets

jet_set = [MJet(E_array[i],px_array[i],py_array[i],pz_array[i]) for i in range(len(E_array))]


In [6]:
def d_ij_SIFT(j1, j2):
  """
  SIFT (Scale-Invariant Filtered Tree) distance metric between two jets, \frac{ ( \Delta m_{ij}^2 ) }{ (E_T^i)^2 + (E_T^j)^2 },
  with E_T^2 = E^2 - p_z^2.

  Parameters:
      j1 (MJet): The first jet.
      j2 (MJet): The second jet.

  Returns:
      float: The distance between the jets.
  """

  m_i_squared = j1.m**2
  m_j_squared = j2.m**2

  E_T_i_squared = j1.E**2 - j1.pz**2
  E_T_j_squared = j2.E**2 - j2.pz**2

  num = abs(m_i_squared - m_j_squared)
  denom = E_T_i_squared + E_T_j_squared

  output = num/denom


  return output


<>:3: SyntaxWarning: invalid escape sequence '\D'
<>:3: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_10902/2306919215.py:3: SyntaxWarning: invalid escape sequence '\D'
  SIFT (Scale-Invariant Filtered Tree) distance metric between two jets, \frac{ ( \Delta m_{ij}^2 ) }{ (E_T^i)^2 + (E_T^j)^2 },


In [7]:
#defining SIFT clustering function

# future: incorporating clustering, drop, isolate conditions
# otherwise, we begin with just a version that brings everything to one big jet

def SIFT(j_initial,thresholding=False, pT_threshold=0.0, eta_threshold=0.0):

  active = list(j_initial)
  final_jets = []

  #main loop
  while active:

    #if only one jet remaining:

    if len(active) == 1:
      final_jets.append(active.pop(0))
      break

    #initializing minimum distance variables
    d_min = float('inf')
    indices_min = None

    for i, j_i in enumerate(active):

      for j in range(i+1, len(active)):

        dij = d_ij_SIFT(j_i, active[j])

        if dij < d_min:
          d_min = dij
          indices_min = (i, j)

    if indices_min is not None:
      idx1, idx2 = sorted(indices_min, reverse=True)
      j_i = active.pop(idx1)
      j_j = active.pop(idx2)

      active.append(j_i + j_j)
    else:
      raise ValueError("No valid clustering found.")

  if thresholding:
    filtered_final_jets = []
    for jet in final_jets:
      if jet.pT >= pT_threshold and abs(jet.eta) <= eta_threshold:
        filtered_final_jets.append(jet)
    return filtered_final_jets
  else:
    return final_jets


In [8]:
clustered_jets = SIFT(jet_set)

In [9]:
#writing output final jets to data file:

file_out = uproot.recreate("FinalJets_SIFT.root")

output_dict = {
    "E": [jet.E for jet in clustered_jets],
    "px": [jet.px for jet in clustered_jets],
    "py": [jet.py for jet in clustered_jets],
    "pz": [jet.pz for jet in clustered_jets],
    "pT": [jet.pT for jet in clustered_jets],
    "eta": [jet.eta for jet in clustered_jets],
    "phi": [jet.phi for jet in clustered_jets],
}

output_df = pd.DataFrame(output_dict)

file_out.mktree("finalJets", output_df)

file_out["finalJets"]



<WritableTree '/finalJets' at 0x7baa112a00b0>

In [10]:
str(clustered_jets[0])
#

"{'E': np.float64(13599.999999999327), 'px': np.float64(1.5051397628251806e-12), 'py': np.float64(-8.959274294673136e-13), 'pz': np.float64(6.729923840431695e-10), 'pT': 1.7516083079585313e-12, 'p': 6.729946635091347e-10, 'phi': -0.5369160919434387, 'eta': 4.9515946898279527e-14, 'm': 13599.999999999327}"